In [1]:
import requests
import mysql.connector
import pandas as pd
from bs4 import BeautifulSoup
from sqlConfig import loginConfig
import tableSetUpMethods as tsum
import time
from concurrent.futures import ThreadPoolExecutor

In [2]:
cnx = mysql.connector.connect(
    user=loginConfig['user'],
    password=loginConfig['password'],
    host=loginConfig['host'],
    database=loginConfig['database']
)
cursor = cnx.cursor()

In [3]:
# Execute the query
query = "SELECT * FROM eventHyperlinks"
cursor.execute(query)
eventHyperLinks = cursor.fetchall()
eventHyperLinks

[(2,
  'UFC Fight Night: Moreno vs. Royval 2',
  'February 24, 2024',
  'Mexico City, Distrito Federal, Mexico',
  'http://ufcstats.com/event-details/902ab9197b83d0db'),
 (3,
  'UFC 298: Volkanovski vs. Topuria',
  'February 17, 2024',
  'Anaheim, California, USA',
  'http://ufcstats.com/event-details/dab0e6cb8c932162'),
 (4,
  'UFC Fight Night: Hermansson vs. Pyfer',
  'February 10, 2024',
  'Las Vegas, Nevada, USA',
  'http://ufcstats.com/event-details/eaea0fc7b76525a8'),
 (5,
  'UFC Fight Night: Dolidze vs. Imavov',
  'February 03, 2024',
  'Las Vegas, Nevada, USA',
  'http://ufcstats.com/event-details/cce79e827569f26e'),
 (6,
  'UFC 297: Strickland vs. Du Plessis',
  'January 20, 2024',
  'Toronto, Ontario, Canada',
  'http://ufcstats.com/event-details/bd85ca0bb4f26cfc'),
 (7,
  'UFC Fight Night: Ankalaev vs. Walker 2',
  'January 13, 2024',
  'Las Vegas, Nevada, USA',
  'http://ufcstats.com/event-details/010986ee359fb863'),
 (8,
  'UFC 296: Edwards vs. Covington',
  'December 16, 

In [4]:
df = pd.DataFrame(eventHyperLinks, columns=['Event ID', 'Event Name', 'Date', 'Location', 'URL'])
df['Date'] = pd.to_datetime(df['Date'], format='%B %d, %Y')
df = df[df['Date'] >= '2000-09-01']

structured_array = df.to_records(index=False)
eventHyperLinks = list(structured_array)

In [5]:
eventHyperLinks

[(2, 'UFC Fight Night: Moreno vs. Royval 2', '2024-02-24T00:00:00.000000000', 'Mexico City, Distrito Federal, Mexico', 'http://ufcstats.com/event-details/902ab9197b83d0db'),
 (3, 'UFC 298: Volkanovski vs. Topuria', '2024-02-17T00:00:00.000000000', 'Anaheim, California, USA', 'http://ufcstats.com/event-details/dab0e6cb8c932162'),
 (4, 'UFC Fight Night: Hermansson vs. Pyfer', '2024-02-10T00:00:00.000000000', 'Las Vegas, Nevada, USA', 'http://ufcstats.com/event-details/eaea0fc7b76525a8'),
 (5, 'UFC Fight Night: Dolidze vs. Imavov', '2024-02-03T00:00:00.000000000', 'Las Vegas, Nevada, USA', 'http://ufcstats.com/event-details/cce79e827569f26e'),
 (6, 'UFC 297: Strickland vs. Du Plessis', '2024-01-20T00:00:00.000000000', 'Toronto, Ontario, Canada', 'http://ufcstats.com/event-details/bd85ca0bb4f26cfc'),
 (7, 'UFC Fight Night: Ankalaev vs. Walker 2', '2024-01-13T00:00:00.000000000', 'Las Vegas, Nevada, USA', 'http://ufcstats.com/event-details/010986ee359fb863'),
 (8, 'UFC 296: Edwards vs. Covi

In [6]:
"""
We had to split the original method in half because it was to intensive and things were getting lost in the middle. Now the first half
is dedicated to taking in a event row from our eventHyperLinks table and vising the page related to that row. From there its going to scrape all
the fights that occurred at this event as well as the winner of that individual fight. This will all be returned in a list of tuples that where each 
individual tuple will represent a fight on the card and will contain things like the winner, link, eventID, eventName, eventDate, eventLocation
"""
def fightStatGrabberA(event):
    time.sleep(1)
    eventID = event[0]
    eventName = event[1]
    eventDate = event[2]
    eventLocation = event[3]
    eventURL = event[4]
    response = requests.get(eventURL)
    soup = BeautifulSoup(response.content, 'html.parser')
    
    #grab main fights table
    tables = soup.find_all('table')
    if not tables:
        print(f"This {eventName} didnt have a table to grab. This was the link {eventURL}")
        return [f"This {eventName} didnt have a table to grab. This was the link {eventURL}"]
    table = tables[0]
    rows = table.find_all('tr')
    
    #grab all onClick links in the fighter table
    fightLinks = []
    winners = []
    for row in rows:
        onclick = row.get('onclick')
        if onclick:
            fightLinks.append(onclick)
    
        #want to find the winner from this table and store for later
        winner_tag = row.find('a', class_='b-link b-link_style_black')
        header_tag = row.find('th')
        if winner_tag and not header_tag:  # Check if the tag exists before calling get_text()
            winner = winner_tag.get_text().strip()
            winners.append(winner)

    #pair
    fightLinks = [fight.split("'")[1] for fight in fightLinks]
    winnerAndLink = list(zip(winners, fightLinks))
    eventStats = [(winner, link, eventID, eventName, eventDate, eventLocation) for winner, link in winnerAndLink]
    return(eventStats)

In [7]:
"""
This is the second part of the original fightStatGrabber method. This is meant to act upon the tuples that are produced by part A.
Its going to take in a tuple labeled eventStats that contains the following information in this exact order:
winner, link, eventID, eventName, eventDate, eventLocation
We're going to grab the link to visit the individual fight page and try to find the top table pertaining to 'Totals' and scrape all of that 
information.
"""
def fightStatGrabberB(eventStats):
    fightStats = []   
    fightLink = eventStats[1]
    time.sleep(1)
    response = requests.get(fightLink)
    soup = BeautifulSoup(response.content, 'html.parser')
        
    tables = soup.find_all('table')
    if tables:
        table = tables[0] 
        stats = table.find_all('p', class_='b-fight-details__table-text')
        currFight = [p.get_text(strip=True) for p in stats]

        #also add in weight class
        weightClass = soup.find("i", class_="b-fight-details__fight-title").get_text(strip=True)
        weight = weightClass.split()[0]
        currFight.append(weight)
        
        currFight.extend(list(eventStats))
        fightStats.append(currFight)
    else:
        return []
    return(fightStats)

In [8]:
with ThreadPoolExecutor(10) as executor:
    winnerGrabber = list(executor.map(fightStatGrabberA, eventHyperLinks))
winnerGrabber = [event for subList in winnerGrabber for event in subList]
print(len(winnerGrabber))

7286


In [9]:
with ThreadPoolExecutor(10) as executor:
    allFighterStats = list(executor.map(fightStatGrabberB, winnerGrabber))
allFighterStats = [event for subList in allFighterStats for event in subList]

In [10]:
#cursor.close()
#cnx.close()

In [27]:
allFighterStats
allFighterStatsDF = pd.DataFrame(allFighterStats, columns=['fighter_A', 'fighter_B', 'fighter_A_KD', 'fighter_B_KD', 'fighter_a_sig_strikes', 'fighter_b_sig_strikes', 'fighter_a_sig_strike_acc', 'fighter_b_sig_strike_acc', 'fighter_a_total_strikes', 'fighter_b_total_strikes', 'fighter_a_takedowns', 'fighter_b_takedowns', 'fighter_a_takedown_acc', 'fighter_b_takedown_acc', 'fighter_a_sub_attempts', 'fighter_b_sub_attempts', 'fighter_a_reversal', 'fighter_b_reversal', 'fighter_a_control_time', 'fighter_b_control_time', 'weight_class', 'winner', 'fightURL', 'eventID', 'event', 'date', 'location'])
allFighterStatsDF['date'] = allFighterStatsDF['date'].astype(str)

In [30]:
cursor.execute("DROP TABLE IF EXISTS fightStats")
cursor.execute("""
    CREATE TABLE fightStats (
        fightID INT AUTO_INCREMENT,
        fighterA VARCHAR(255),
        fighterB VARCHAR(255),
        fighter_A_KD VARCHAR(255),
        fighter_B_KD VARCHAR(255),
        fighter_a_sig_strikes VARCHAR(255),
        fighter_b_sig_strikes VARCHAR(255),
        fighter_a_sig_strike_acc VARCHAR(255),
        fighter_b_sig_strike_acc VARCHAR(255),
        fighter_a_total_strikes VARCHAR(255),
        fighter_b_total_strikes VARCHAR(255),
        fighter_a_takedowns VARCHAR(255),
        fighter_b_takedowns VARCHAR(255),
        fighter_a_takedown_acc VARCHAR(255),
        fighter_b_takedown_acc VARCHAR(255),
        fighter_a_sub_attempts VARCHAR(255),
        fighter_b_sub_attempts VARCHAR(255),
        fighter_a_reversal VARCHAR(255),
        fighter_b_reversal VARCHAR(255),
        fighter_a_control_time VARCHAR(255),
        fighter_b_control_time VARCHAR(255),
        weight_class VARCHAR(255),
        winner VARCHAR(255),
        fightURL VARCHAR(255),
        eventID INT,
        event VARCHAR(255),
        date VARCHAR(255),
        location VARCHAR(255),
        PRIMARY KEY (fightID),
        FOREIGN KEY (eventID) REFERENCES eventHyperlinks(eventID)
    )
""")

# Insert the data
query = "INSERT INTO fightStats (fighterA, fighterB, fighter_A_KD, fighter_B_KD, fighter_a_sig_strikes, fighter_b_sig_strikes, fighter_a_sig_strike_acc, fighter_b_sig_strike_acc, fighter_a_total_strikes, fighter_b_total_strikes, fighter_a_takedowns, fighter_b_takedowns, fighter_a_takedown_acc, fighter_b_takedown_acc, fighter_a_sub_attempts, fighter_b_sub_attempts, fighter_a_reversal, fighter_b_reversal, fighter_a_control_time, fighter_b_control_time, weight_class, winner, fightURL, eventID, event, date, location) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)"
allFighterStatsList = allFighterStatsDF.to_records(index=False).tolist()
cursor.executemany(query, allFighterStatsList)
cnx.commit()

In [25]:
query = """
DESCRIBE eventHyperlinks;
"""
cursor.execute(query)
test = cursor.fetchall()
test

[('eventID', 'int', 'NO', 'PRI', None, 'auto_increment'),
 ('eventName', 'varchar(255)', 'YES', '', None, ''),
 ('eventDate', 'varchar(255)', 'YES', '', None, ''),
 ('eventLocation', 'varchar(255)', 'YES', '', None, ''),
 ('eventHyperLink', 'varchar(255)', 'YES', '', None, '')]

In [31]:
cursor.close()
cnx.close()